# Ecommerce ETL Data Cleaning

This notebook is used to explore and clean the data before writing the ETL script.

In [1]:
import pandas as pd

### Extract

In [2]:
# Data paths
customer_path = r'datasets\customers.csv'
orders_path = r'datasets\orders.csv'
products_path = r'datasets\products.csv'
order_items_path = r'datasets\order_items.csv'

# Import data into dataframes
customers_df = pd.read_csv(customer_path)
orders_df = pd.read_csv(orders_path)
products_df = pd.read_csv(products_path)
order_items_df = pd.read_csv(order_items_path)

### Transform

In [3]:
# Remove duplicate rows for any dataset
def remove_duplicates(df):
    before = len(df)
    df = df.drop_duplicates()
    after = len(df)
   
    print(f"Removed {before - after} duplicate rows.")
    
    return df

In [ ]:
# Remove rows with missing values based on index
def drop_missing_values(df, indexes):
    if indexes == 0:
        # Remove all rows with missing values in the dataset
        df = df.dropna()
    else:
        

In [4]:
# Customers
customers_df.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [30]:
# Select necessary columns
customers = customers_df[['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']]

In [42]:
# Remove duplicates
customers = remove_duplicates(customers)

Removed 0 duplicate rows.


In [56]:
# Handle missing values
customers.isna().sum()

# If any column has a missing value remove row
before_dropna = len(customers)
customers.dropna(inplace=True)
after_dropna = len(customers)
print(f"Removed {before_dropna - after_dropna} rows with missing values.")

Removed 0 rows with missing values.


In [48]:
# Standardize formats
customers['customer_city'] = customers['customer_city'].str.title()

In [4]:
# Customers ETL Pipeline Code
print("Transforming customers dataset...")

# Select relevant columns
customers = customers_df[['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']]

print(f"Rows before transformation: {len(customers)}.")

# Remove duplicates
customers = remove_duplicates(customers)

# Handle missing values
before_dropna = len(customers)
customers = customers.dropna()
after_dropna = len(customers)
print(f"Removed {before_dropna - after_dropna} rows with missing values.")

# Capitalize customer_city column 
customers['customer_city'] = customers['customer_city'].str.title()

print(f"Rows after transformation: {len(customers)}.")

Transforming customers dataset...
Rows before transformation: 99441.
Removed 0 duplicate rows.
Removed 0 rows with missing values.
Rows after transformation: 99441.


In [7]:
# Orders
orders_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [62]:
# Relevant columns
orders = orders_df[
    [
        'order_id', 
        'customer_id', 
        'order_status', 
        'order_purchase_timestamp', 
        'order_delivered_customer_date']
]

In [63]:
# Duplicates
orders = remove_duplicates(orders)

Removed 0 duplicate rows.


In [75]:
# Missing values
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_delivered_customer_date    2965
dtype: int64

In [88]:
# Order has not been delivered yet.
orders[orders['order_delivered_customer_date'].isna()]['order_status'].value_counts()

shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: order_status, dtype: int64

In [110]:
# Drop rows with order status = delivered and no delivery date
filter_na = orders['order_delivered_customer_date'].isna()
filter_delivered = orders['order_status'] == 'delivered'

indexes = orders[filter_na & filter_delivered].index
orders.drop(indexes, inplace=True)

In [111]:
# Standardize formats
orders['order_status'] = orders['order_status'].str.title()

In [117]:
# Convert dates to timestamps
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])

In [118]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 99433 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99433 non-null  object        
 1   customer_id                    99433 non-null  object        
 2   order_status                   99433 non-null  object        
 3   order_purchase_timestamp       99433 non-null  datetime64[ns]
 4   order_delivered_customer_date  96476 non-null  datetime64[ns]
dtypes: datetime64[ns](2), object(3)
memory usage: 4.6+ MB


In [119]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,Delivered,2017-10-02 10:56:33,2017-10-10 21:25:13
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,Delivered,2018-07-24 20:41:37,2018-08-07 15:27:45
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,Delivered,2018-08-08 08:38:49,2018-08-17 18:06:29
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,Delivered,2017-11-18 19:28:06,2017-12-02 00:28:42
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,Delivered,2018-02-13 21:18:39,2018-02-16 18:17:02


In [5]:
# Orders ETL Pipeline Code

# Relevant columns
orders = orders_df[[
    'order_id', 
    'customer_id', 
    'order_status', 
    'order_purchase_timestamp', 
    'order_delivered_customer_date'
]]

print(f"Rows before transformation: {len(orders)}.")

# Duplicates
orders = remove_duplicates(orders)

# Drop rows with order status = delivered and no delivery date
filter_na = orders['order_delivered_customer_date'].isna()
filter_delivered = orders['order_status'] == 'delivered'
indexes = orders[filter_na & filter_delivered].index
before_dropna = len(orders)
orders = orders.drop(indexes)
after_dropna = len(orders)
print(f"Removed {before_dropna - after_dropna} rows with missing values.")

# Standardize formats
orders['order_status'] = orders['order_status'].str.title()

# Convert dates to timestamps
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])

print(f"Rows after transformation: {len(orders)}.")

Rows before transformation: 99441.
Removed 0 duplicate rows.
Removed 8 rows with missing values.
Rows after transformation: 99433.


In [9]:
# Products
products_df.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [16]:
# Relevant columns
products = products_df[['product_id', 'product_category_name']]

In [12]:
# Remove duplicates
products = remove_duplicates(products)

Removed 0 duplicate rows.


In [8]:
# Handle missing values
products.isna().sum()

# Remove rows with missing product category 
indexes = products[products['product_category'].isna()].index
before_dropna = len(products)
products = products.drop(indexes)
after_dropna = len(products)
print(f"Removed {before_dropna - after_dropna} rows with missing values.")

Removed 0 rows with missing values.


In [6]:
# Products ETL Pipeline Code

# Select relevant columns
products = products_df[['product_id', 'product_category_name']]

print(f"Rows before transformation: {len(products)}.")

# Rename columns
products = products.rename(columns={"product_category_name": "product_category"})

# Remove duplicates
products = remove_duplicates(products)

# Remove rows with missing product category 
indexes = products[products['product_category'].isna()].index
before_dropna = len(products)
products = products.drop(indexes)
after_dropna = len(products)
print(f"Removed {before_dropna - after_dropna} rows with missing values.")

# Replace underscores in product_category with spaces
products['product_category'] = products['product_category'].str.replace("_", " ").str.strip().str.title()

print(f"Rows after transformation: {len(products)}.")

Rows before transformation: 32951.
Removed 0 duplicate rows.
Removed 610 rows with missing values.
Rows after transformation: 32341.


In [13]:
# Order Items
order_items_df.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [19]:
# Relevant columns
order_items = order_items_df[[
    'order_id',
    'order_item_id',
    'product_id',
    'seller_id',
    'price',
    'freight_value'
]]

In [21]:
# Remove duplicates
order_items = remove_duplicates(order_items)

Removed 0 duplicate rows.


In [25]:
# Handle missing data
order_items.isna().sum() 

# Drop rows with missing values
order_items = order_items.dropna()

In [35]:
# Validate numeric ranges 
order_items.describe()

# Accept rows where price and freight value > 0
order_items = order_items[order_items['price'] > 0]
order_items = order_items[order_items['freight_value'] > 0]

In [8]:
# Order Items ETL Pipeline Code

# Select relevant columns
order_items = order_items_df[[
    'order_id',
    'order_item_id',
    'product_id',
    'seller_id',
    'price',
    'freight_value'
]]

print(f"Rows before transformation: {len(order_items)}.")

# Remove duplicates
order_items = remove_duplicates(order_items)

# Drop rows with missing values
before_dropna = len(order_items)
order_items = order_items.dropna()
after_dropna = len(order_items)
print(f"Removed {before_dropna - after_dropna} rows with missing values.")

# Accept rows where price and freight value > 0
order_items = order_items[order_items['price'] > 0]
after_price_filter = len(order_items)
print(f"Removed {after_dropna - after_price_filter} rows with price < 0.")

order_items = order_items[order_items['freight_value'] > 0]
after_freight_filter = len(order_items)
print(f"Removed {after_price_filter - after_freight_filter} rows with freight value < 0.")

print(f"Rows after transformation: {len(order_items)}.")

Rows before transformation: 112650.
Removed 0 duplicate rows.
Removed 0 rows with missing values.
Removed 0 rows with price < 0.
Removed 383 rows with freight value < 0.
Rows after transformation: 112267.


In [12]:
order_items.dtypes

order_id          object
order_item_id      int64
product_id        object
seller_id         object
price            float64
freight_value    float64
dtype: object

In [1]:
my_dict = {
    "lol": 1,
    "yo": 2
}

In [3]:
my_dict['lol']

1